# Rigshospitalet Operating Room Efficiency Analysis

**Course:** From Analytics to Action (42577) — Spring 2026  
**Data:** Completed operations from Rigshospitalet, 2024

This notebook investigates three operational findings that represent actionable opportunities to reduce delays and improve OR throughput — none of which require clinical changes.

---

## Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# Rigshospitalet brand color
RH_BLUE = '#0077B6'
RH_LIGHT = '#90E0EF'
ACCENT_RED = '#E63946'
ACCENT_GREEN = '#2A9D8F'

print('Setup complete.')

In [ ]:
# ============================================================
# LOAD DATA — update the path to your CSV file
# ============================================================
DATA_PATH = 'completed_operations.csv'  # <-- change this to your file path

df = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8')
print(f'Loaded {len(df):,} rows × {len(df.columns)} columns')
df.head(3)

## Data Preparation

Parse timestamps, extract procedure names, and create derived columns we'll need across all three findings.

In [ ]:
# --- Parse key datetime columns ---
time_cols = [
    'Dato',
    'Pt ankommet til hospitalet',
    'Planlagt stue klargøring start',
    'Stue klargøring start',
    'Stue klargjort',
    'Patient på stuen (Planlagt)',
    'Patient på stuen',
    'Anæstesistart',
    'Anæstesi melder klar',
    'Procedure start',
    'Procedure slut',
    'Patient klar til afgang',
    'Patient forlader stuen (Planlagt)',
    'Patient forlader stuen',
]

for col in time_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')

# --- Numeric delay columns ---
df['Forsinkelse (minutter)'] = pd.to_numeric(df['Forsinkelse (minutter)'], errors='coerce')
df['Overskredet (minutter)'] = pd.to_numeric(df['Overskredet (minutter)'], errors='coerce')

# --- Extract procedure name (strip the ID in brackets) ---
df['Procedure_Name'] = (
    df['Procedure - Tekst & ID']
    .str.replace(r'\s*\[.*\]', '', regex=True)
    .str.strip()
)

# --- Derived time features ---
df['Day_of_Week'] = df['Dato'].dt.day_name()
df['Month'] = df['Dato'].dt.month
df['Is_Acute'] = df['Akut case (J/N)'].map({'Ja': True, 'Nej': False})

# --- Procedure duration (minutes) ---
df['Procedure_Duration_Min'] = (
    (df['Procedure slut'] - df['Procedure start']).dt.total_seconds() / 60
)

# --- Unique case count (each Case-ID is one operation, but may have multiple procedure rows) ---
n_cases = df['Case-ID Anonymous'].nunique()
print(f'Unique operations (Case-IDs): {n_cases:,}')
print(f'Total procedure rows: {len(df):,}')
print(f'Date range: {df["Dato"].min().date()} to {df["Dato"].max().date()}')

In [ ]:
# Quick sanity check on delay distribution
print('Delay (Forsinkelse) summary:')
print(df['Forsinkelse (minutter)'].describe().round(1))
print(f'\nMissing delay values: {df["Forsinkelse (minutter)"].isna().sum():,}')

---

## Finding 1: SEPTUMPLASTIK — 55-Minute Room Variation

**The strange thing:** The exact same elective procedure runs 55 minutes differently depending on which operating room it's assigned to. No difference in surgeon, patient complexity, or anything clinical — purely operational.

**Why it matters:** If the best rooms prove it can be done efficiently, the variation is fixable through operational standardization.

**Potential impact:** ~455 hours/year saved

In [ ]:
# Filter to septumplastik cases
septum = df[df['Procedure_Name'].str.contains('SEPTUMPLASTIK', case=False, na=False)].copy()

print(f'SEPTUMPLASTIK rows: {len(septum):,}')
print(f'Unique cases: {septum["Case-ID Anonymous"].nunique()}')
print(f'Rooms used: {septum["Stue"].nunique()}')
print(f'Acute cases: {septum["Is_Acute"].sum()} ({septum["Is_Acute"].mean()*100:.0f}%)')

In [ ]:
# Delay by room — only rooms with enough cases to be meaningful
MIN_CASES = 5  # minimum cases per room to include

room_stats = (
    septum
    .groupby('Stue')['Forsinkelse (minutter)']
    .agg(['mean', 'median', 'std', 'count'])
    .rename(columns={'mean': 'Mean Delay', 'median': 'Median Delay',
                     'std': 'Std', 'count': 'Cases'})
    .sort_values('Mean Delay', ascending=True)
)

room_stats_filtered = room_stats[room_stats['Cases'] >= MIN_CASES]

print(f'Rooms with >= {MIN_CASES} cases: {len(room_stats_filtered)}')
print(f'\nBest room:  {room_stats_filtered.index[0]} '
      f'(mean delay: {room_stats_filtered["Mean Delay"].iloc[0]:+.0f} min, '
      f'n={room_stats_filtered["Cases"].iloc[0]:.0f})')
print(f'Worst room: {room_stats_filtered.index[-1]} '
      f'(mean delay: {room_stats_filtered["Mean Delay"].iloc[-1]:+.0f} min, '
      f'n={room_stats_filtered["Cases"].iloc[-1]:.0f})')
print(f'\nVariation (worst − best): '
      f'{room_stats_filtered["Mean Delay"].iloc[-1] - room_stats_filtered["Mean Delay"].iloc[0]:.0f} min')

room_stats_filtered.round(1)

In [ ]:
# --- Visualization: Mean delay by room ---
fig, ax = plt.subplots(figsize=(12, 6))

colors = [ACCENT_GREEN if v <= 0 else ACCENT_RED if v == room_stats_filtered['Mean Delay'].max()
          else RH_BLUE for v in room_stats_filtered['Mean Delay']]

bars = ax.barh(
    room_stats_filtered.index,
    room_stats_filtered['Mean Delay'],
    color=colors,
    edgecolor='white',
    linewidth=0.5
)

# Add case count labels
for bar, n in zip(bars, room_stats_filtered['Cases']):
    w = bar.get_width()
    ax.text(w + 1 if w >= 0 else w - 1, bar.get_y() + bar.get_height()/2,
            f'n={n:.0f}', va='center', ha='left' if w >= 0 else 'right',
            fontsize=9, color='#555')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean Delay (minutes)')
ax.set_title('Finding 1: SEPTUMPLASTIK — Mean Delay by Operating Room',
             fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('finding1_septum_room_variation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Box plot to show the full distribution per room ---
fig, ax = plt.subplots(figsize=(12, 6))

room_order = room_stats_filtered.index.tolist()
septum_plot = septum[septum['Stue'].isin(room_order)]

sns.boxplot(
    data=septum_plot, y='Stue', x='Forsinkelse (minutter)',
    order=room_order, palette='coolwarm', ax=ax,
    flierprops=dict(marker='o', markersize=3, alpha=0.4)
)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Delay (minutes)')
ax.set_ylabel('')
ax.set_title('SEPTUMPLASTIK — Delay Distribution by Room', fontweight='bold')
plt.tight_layout()
plt.savefig('finding1_septum_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Impact estimation ---
# If all rooms performed at the level of the best room, how many hours would be saved?

best_room_mean = room_stats_filtered['Mean Delay'].iloc[0]
total_excess_min = 0

for room, row in room_stats_filtered.iterrows():
    excess_per_case = row['Mean Delay'] - best_room_mean
    if excess_per_case > 0:
        total_excess_min += excess_per_case * row['Cases']

print(f'If all rooms matched the best room ({room_stats_filtered.index[0]}):') 
print(f'  Total excess delay in data: {total_excess_min:,.0f} minutes')
print(f'  = {total_excess_min/60:,.0f} hours')
print(f'  Annualized (if this data spans <12 months, scale accordingly)')

### Finding 1 — Takeaway

The variation is **not clinical** — it's operational. The best-performing rooms prove that SEPTUMPLASTIK can run on time or ahead of schedule. The recommended action is to audit what the best rooms do differently in terms of equipment staging, turnover protocol, and team assignment, then replicate those practices.

---

## Finding 2: BRONKOSKOPI MED BAL — 115-Minute Day-of-Week Swing

**The strange thing:** The same procedure varies by 115 minutes in average delay depending purely on what day of the week it falls on.

**Pattern:** Monday runs 16 min early. Sunday runs 99 min late. This points directly at weekend staffing and staging issues.

In [ ]:
# Filter to bronkoskopi med BAL cases
bronk = df[df['Procedure_Name'].str.contains('BRONKOSKOPI MED BAL', case=False, na=False)].copy()

print(f'BRONKOSKOPI MED BAL rows: {len(bronk):,}')
print(f'Unique cases: {bronk["Case-ID Anonymous"].nunique()}')
print(f'Acute: {bronk["Is_Acute"].sum()} ({bronk["Is_Acute"].mean()*100:.0f}%)')

In [ ]:
# Day-of-week analysis
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow_stats = (
    bronk
    .groupby('Day_of_Week')['Forsinkelse (minutter)']
    .agg(['mean', 'median', 'std', 'count'])
    .rename(columns={'mean': 'Mean Delay', 'median': 'Median Delay',
                     'std': 'Std', 'count': 'Cases'})
    .reindex(day_order)
    .dropna(subset=['Cases'])
)

print('Mean delay by day of week:')
print(dow_stats.round(1))
print(f'\nSwing (worst − best): '
      f'{dow_stats["Mean Delay"].max() - dow_stats["Mean Delay"].min():.0f} min')

In [ ]:
# --- Visualization: Day-of-week bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))

colors_dow = [ACCENT_GREEN if v <= 0 else ACCENT_RED if v == dow_stats['Mean Delay'].max()
              else RH_BLUE for v in dow_stats['Mean Delay']]

bars = ax.bar(dow_stats.index, dow_stats['Mean Delay'], color=colors_dow,
              edgecolor='white', linewidth=0.5)

# Case count annotations
for bar, n in zip(bars, dow_stats['Cases']):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 2 if h >= 0 else h - 5,
            f'n={n:.0f}', ha='center', va='bottom' if h >= 0 else 'top',
            fontsize=9, color='#555')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean Delay (minutes)')
ax.set_title('Finding 2: BRONKOSKOPI MED BAL — Mean Delay by Day of Week',
             fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('finding2_bronk_dayofweek.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Scatter: individual cases colored by weekday vs weekend ---
bronk['Is_Weekend'] = bronk['Day_of_Week'].isin(['Saturday', 'Sunday'])

fig, ax = plt.subplots(figsize=(12, 5))
for is_wknd, label, color in [(False, 'Weekday', RH_BLUE), (True, 'Weekend', ACCENT_RED)]:
    sub = bronk[bronk['Is_Weekend'] == is_wknd]
    ax.scatter(sub['Dato'], sub['Forsinkelse (minutter)'],
              alpha=0.5, s=30, color=color, label=label, edgecolors='white', linewidth=0.3)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Date')
ax.set_ylabel('Delay (minutes)')
ax.set_title('BRONKOSKOPI MED BAL — Individual Case Delays Over Time', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('finding2_bronk_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### Finding 2 — Takeaway

The Monday-to-Sunday swing suggests a **staffing and staging issue**, not a clinical one. The recommended action is to audit Sunday staffing levels, seniority mix, and equipment pre-staging compared to Monday. The Monday "reset" to negative delays implies the weekend backlog clears overnight.

---

## Finding 3: ESOPHAGOSKOPI — Planned vs. Acute, 42-Minute Swing

**The strange thing:** Planned esophagoscopies run ~21 min *ahead* of schedule. Acute ones run ~21 min *behind*. Same procedure, 42-minute gap — driven entirely by pre-operative readiness.

In [ ]:
# Filter to esophagoskopi (various spellings in the data)
eso = df[df['Procedure_Name'].str.contains('ESOPHAGO|ESOFAGO|ØSOFAGO', case=False, na=False)].copy()

print(f'ESOPHAGOSKOPI-related rows: {len(eso):,}')
print(f'Unique cases: {eso["Case-ID Anonymous"].nunique()}')
print(f'\nBreakdown by acute/planned:')
print(eso.groupby('Is_Acute')['Forsinkelse (minutter)'].agg(['mean', 'median', 'count']).round(1))

In [ ]:
# --- Visualization: Acute vs Planned comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: bar chart of mean delay
ax = axes[0]
groups = eso.groupby('Is_Acute')['Forsinkelse (minutter)'].agg(['mean', 'count'])
groups.index = ['Planned (Nej)', 'Acute (Ja)']
bar_colors = [ACCENT_GREEN, ACCENT_RED]
bars = ax.bar(groups.index, groups['mean'], color=bar_colors, edgecolor='white', width=0.5)
for bar, (_, row) in zip(bars, groups.iterrows()):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1 if h >= 0 else h - 3,
            f'{h:+.0f} min\n(n={row["count"]:.0f})', ha='center',
            va='bottom' if h >= 0 else 'top', fontsize=10)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean Delay (minutes)')
ax.set_title('Mean Delay: Planned vs Acute', fontweight='bold')

# Right panel: overlapping histograms
ax = axes[1]
for is_acute, label, color in [(False, 'Planned', ACCENT_GREEN), (True, 'Acute', ACCENT_RED)]:
    sub = eso[eso['Is_Acute'] == is_acute]['Forsinkelse (minutter)'].dropna()
    if len(sub) > 0:
        ax.hist(sub, bins=20, alpha=0.5, color=color, label=label, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Delay (minutes)')
ax.set_ylabel('Count')
ax.set_title('Delay Distribution: Planned vs Acute', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('finding3_eso_acute_vs_planned.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Dig deeper: Where does the acute delay accumulate? ---
# Compare the gap between arrival and procedure start for acute vs planned

eso['Arrival_to_Proc_Start'] = (
    (eso['Procedure start'] - eso['Pt ankommet til hospitalet']).dt.total_seconds() / 60
)

print('Minutes from patient arrival to procedure start:')
print(eso.groupby('Is_Acute')['Arrival_to_Proc_Start']
      .agg(['mean', 'median', 'count']).round(1))
print('\n→ The gap here shows where the extra time goes for acute cases.')

In [ ]:
# --- Timeline comparison: mean timestamps for planned vs acute ---
milestones = [
    ('Pt ankommet til hospitalet', 'Arrival'),
    ('Patient på stuen', 'Patient in OR'),
    ('Procedure start', 'Procedure Start'),
    ('Procedure slut', 'Procedure End'),
    ('Patient forlader stuen', 'Patient Leaves OR'),
]

timeline_data = []
for is_acute in [False, True]:
    sub = eso[eso['Is_Acute'] == is_acute]
    label = 'Acute' if is_acute else 'Planned'
    # Use relative time from arrival
    for col, name in milestones:
        diff = (sub[col] - sub['Pt ankommet til hospitalet']).dt.total_seconds() / 60
        timeline_data.append({
            'Type': label,
            'Milestone': name,
            'Mean Minutes from Arrival': diff.mean()
        })

tl = pd.DataFrame(timeline_data)

fig, ax = plt.subplots(figsize=(12, 4))
for typ, color in [('Planned', ACCENT_GREEN), ('Acute', ACCENT_RED)]:
    sub = tl[tl['Type'] == typ]
    ax.plot(sub['Milestone'], sub['Mean Minutes from Arrival'],
            marker='o', markersize=10, linewidth=2.5, color=color, label=typ)
    for _, row in sub.iterrows():
        ax.annotate(f'{row["Mean Minutes from Arrival"]:.0f} min',
                    (row['Milestone'], row['Mean Minutes from Arrival']),
                    textcoords='offset points', xytext=(0, 12),
                    ha='center', fontsize=9, color=color)

ax.set_ylabel('Mean Minutes from Patient Arrival')
ax.set_title('ESOPHAGOSKOPI — Process Timeline: Planned vs Acute', fontweight='bold')
ax.legend()
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('finding3_eso_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

### Finding 3 — Takeaway

Planned cases arrive prepared: fasting confirmed, consent signed, equipment pre-set. Acute cases arrive unprepared, and the rushed intake adds 20–30 minutes before the case even starts. The recommended action is to add a **30-minute structured intake buffer** before acute esophagoscopy slots — running the same pre-op checklist as planned cases.

---

## Summary of Findings

| # | Procedure | Issue | Root Cause | Recommended Action | Est. Impact |
|---|-----------|-------|------------|-------------------|-------------|
| 1 | SEPTUMPLASTIK | 55-min room variation | Operational differences between rooms | Audit & replicate best-room practices | ~455 hrs/yr |
| 2 | BRONKOSKOPI MED BAL | 115-min day-of-week swing | Weekend staffing/staging gaps | Audit Sunday vs Monday staffing | TBD |
| 3 | ESOPHAGOSKOPI | 42-min acute vs planned gap | Unprepared acute intake | 30-min structured intake buffer | TBD |

All three findings share a common theme: **the delays are operational, not clinical**. The procedures themselves are the same — what differs is the context in which they run.

In [ ]:
print('Notebook complete. Charts saved:')
print('  - finding1_septum_room_variation.png')
print('  - finding1_septum_boxplot.png')
print('  - finding2_bronk_dayofweek.png')
print('  - finding2_bronk_scatter.png')
print('  - finding3_eso_acute_vs_planned.png')
print('  - finding3_eso_timeline.png')